# Passive compliance — weight loading

How the hand's **torsional (joint-space) spring stiffness** determines the
steady-state finger deflection under hanging weights.  The four fingers are
commanded to hold `WEIGHT_POSE`; a weight pulls them toward extension.
Higher stiffness → smaller angular deviation from the target for the same load.

**Expected relationship** (ideal torsional spring):  
$$\Delta\theta = \frac{F \cdot L}{K}$$
where $F = mg$, $L$ is the effective finger length, and $K$ is the joint stiffness.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('../../'))
from hand_config import (
    STIFFNESS_CONDITIONS, WEIGHTS_G, WEIGHT_POSE,
    WEIGHT_FINGERS, WEIGHT_LOG_DURATION,
)

import shutil
if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False

COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
OUTPUT = os.path.join('outputs', 'weight_compliance_hand')
os.makedirs(OUTPUT, exist_ok=True)

# Motor index of the MCP joint for each finger
FINGER_MCP_IDX = {'index': 5, 'middle': 7, 'ring': 9, 'pinky': 11}
TARGET_MCP_DEG = np.rad2deg(WEIGHT_POSE[0])   # MCP target in degrees

def _synth(k):
    """Synthetic demo data: deflection scales with weight and 1/K."""
    rng = np.random.default_rng(int(k * 1000))
    g = 9.81; L = 0.08   # [m] effective finger length
    rows = []
    t = 0.0; dt = 0.02
    n = int(WEIGHT_LOG_DURATION / dt)
    cols = [f'q_{i}' for i in range(13)] + [f'qdot_{i}' for i in range(13)] + [f'tau_{i}' for i in range(13)]
    for w_g in WEIGHTS_G:
        delta = (w_g * 1e-3 * g * L) / k   # [rad]
        for _ in range(n):
            q = np.zeros(13)
            for fname, midx in FINGER_MCP_IDX.items():
                q[midx]   = max(0.0, WEIGHT_POSE[0] - delta + rng.normal(0, 0.004))
                q[midx+1] = max(0.0, WEIGHT_POSE[1] - delta * 0.6 + rng.normal(0, 0.004))
            row = {'time_s': f'{t:.3f}', 'k_rot': k, 'weight_g': w_g}
            row.update({f'q_{j}': f'{q[j]:.5f}' for j in range(13)})
            row.update({f'qdot_{j}': '0' for j in range(13)})
            row.update({f'tau_{j}': '0' for j in range(13)})
            rows.append(row); t += dt
    return pd.DataFrame(rows)

DEMO = False
def load(k):
    global DEMO
    path = os.path.join(OUTPUT, f'data_K{k:.3f}.csv')
    if os.path.isfile(path):
        return pd.read_csv(path)
    DEMO = True
    return _synth(k)

DATA = {k: load(k) for k in STIFFNESS_CONDITIONS}
banner = ' DEMO (synthetic) data ' if DEMO else ' measured data '
print(f'Loaded{banner}|  K levels: {STIFFNESS_CONDITIONS}')

## Deviation from target vs weight

Mean MCP angular deviation ($\theta_\mathrm{target} - \theta_\mathrm{actual}$, in degrees)
as a function of hanging weight, averaged across the four fingers.  Error bars show
the standard deviation across the logging window.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for i, k in enumerate(STIFFNESS_CONDITIONS):
    d   = DATA[k].copy()
    col = COLORS[i % len(COLORS)]
    ws, means, stds = [], [], []

    for w in WEIGHTS_G:
        rows = d[d['weight_g'] == w]
        # average deviation across all four fingers (MCP only)
        deviations = []
        for fname, midx in FINGER_MCP_IDX.items():
            q_mcp = rows[f'q_{midx}'].astype(float)
            dev   = np.rad2deg(WEIGHT_POSE[0]) - np.rad2deg(q_mcp)
            deviations.append(dev.values)
        dev_all = np.concatenate(deviations)
        ws.append(w); means.append(dev_all.mean()); stds.append(dev_all.std())

    ws, means, stds = np.array(ws), np.array(means), np.array(stds)
    ax.plot(ws, means, 'o-', color=col, lw=2.0,
            label=rf'$K = {k:.2f}$ N$\cdot$m/rad')
    ax.fill_between(ws, means - stds, means + stds, color=col, alpha=0.15)

ax.set_xlabel('Hanging weight [g]')
ax.set_ylabel(r'MCP deviation from target [deg]')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left', title='torsional stiffness')
ax.grid(True, lw=0.4, alpha=0.5)
if DEMO:
    fig.text(0.5, 0.5, 'DEMO', fontsize=90, color='0.92',
             ha='center', va='center', rotation=30, zorder=0)
fig.savefig(os.path.join(OUTPUT, 'weight_deviation_vs_weight.pdf'))
plt.show()

## Per-finger deviation

Same metric broken down by finger.  Differences across fingers reveal any
asymmetry in the finger geometry or motor efficiency.

In [ ]:
fig, axes = plt.subplots(1, len(WEIGHT_FINGERS), figsize=(13, 5), sharey=True)

for ax, fname in zip(axes, WEIGHT_FINGERS):
    midx = FINGER_MCP_IDX[fname]
    for i, k in enumerate(STIFFNESS_CONDITIONS):
        d   = DATA[k].copy()
        col = COLORS[i % len(COLORS)]
        ws, means = [], []
        for w in WEIGHTS_G:
            rows  = d[d['weight_g'] == w]
            q_mcp = rows[f'q_{midx}'].astype(float)
            means.append((np.rad2deg(WEIGHT_POSE[0]) - np.rad2deg(q_mcp)).mean())
            ws.append(w)
        ax.plot(ws, means, 'o-', color=col, lw=1.8,
                label=rf'$K={k:.2f}$')
    ax.set_title(fname)
    ax.set_xlabel('Weight [g]')
    ax.grid(True, lw=0.4, alpha=0.5)

axes[0].set_ylabel('MCP deviation [deg]')
axes[0].set_ylim(bottom=0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', title='stiffness [N·m/rad]',
           fontsize='small', ncol=1)
if DEMO:
    fig.text(0.5, 0.5, 'DEMO', fontsize=90, color='0.92',
             ha='center', va='center', rotation=30, zorder=0)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT, 'weight_deviation_per_finger.pdf'))
plt.show()